In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

c:\Users\SELVAGANAPATHI\anaconda3\envs\tune\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",        # NF4 (important)
    bnb_4bit_use_double_quant=False,  # Double quantization
    bnb_4bit_compute_dtype=torch.float16
)

In [3]:
import accelerate
print(accelerate.__version__)

1.13.0


In [4]:
from huggingface_hub import login
from dotenv import load_dotenv
import os

load_dotenv()
token = os.getenv("HF_TOKEN")
if token:
    login(token)
else:
    print("HF_TOKEN not found in .env file")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [5]:
from huggingface_hub import logging
logging.set_verbosity_info()

In [ ]:
# model_name = "LiquidAI/LFM2.5-350M"

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map="auto"
# )

c:\Users\SELVAGANAPATHI\anaconda3\envs\tune\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SELVAGANAPATHI\.cache\huggingface\hub\models--LiquidAI--LFM2.5-350M. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 148/148 [00:01<00:00, 84.42it/s] 


In [ ]:
# tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# inputs = tokenizer("What is the capital of France?", return_tensors="pt").to(model.device)
# outputs = model.generate(**inputs, max_length=50)
# print(tokenizer.decode(outputs[0], skip_special_tokens=True))

What is the capital of France?

A) Paris

B) Rome

C) London

D) Berlin

Answer: A


In [ ]:
# print(outputs)

tensor([[    1,  3493,   856,   779,  5706,   803,  4481,   540,   509,   542,
           518,  5242,   509,   543,   518, 10000,   509,   544,   518,  5096,
           509,   545,   518,  7053,   509, 40131,   535,   835,     7]],
       device='cuda:0')


In [6]:
model_name = "LiquidAI/LFM2.5-350M"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    local_files_only=True
)

Loading weights: 100%|██████████| 148/148 [00:01<00:00, 110.85it/s]


In [7]:
tokenizer.pad_token = tokenizer.eos_token

What Just Happened (IMPORTANT)

You just:

✅ Loaded a model
✅ Compressed it to 4-bit (NF4)
✅ Reduced memory drastically
✅ Kept it ready for training

Appling loRA

In [8]:
from peft import LoraConfig, get_peft_model

In [9]:
lora_config = LoraConfig(
    r=64,                     # rank
    lora_alpha=128,            # scaling
    target_modules=["q_proj", "v_proj"],  # attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,376,256 || all params: 355,860,224 || trainable%: 0.3867


In [10]:
dataset = [
    # GREETINGS & BASIC INTERACTION (Variations)
    {"text": "### Instruction:\nSay hello casually\n### Response:\nVada macha! Epdi iruka da bro 😄"},
    {"text": "### Instruction:\nSay hello\n### Response:\nHey machi! Enna da scene? Long time no see! 👋"},
    {"text": "### Instruction:\nGreet your friend\n### Response:\nDei mama! Vada vada, miss pannen da unna 😎"},
    {"text": "### Instruction:\nSay good morning\n### Response:\nGood morning macha! Coffee kudichiya? ☕️"},
    {"text": "### Instruction:\nSay good night\n### Response:\nGood night da, kalam meet pannalam. Sweet dreams! 🌙"},
    {"text": "### Instruction:\nAsk how are you\n### Response:\nDei, epdi da going? Life elam smooth ah? 🤔"},
    
    # TECH EXPLANATIONS (AI, Programming, etc.)
    {"text": "### Instruction:\nExplain AI simply\n### Response:\nAI-na macha, computer ku brain kudutha madhiri da. Ippo ellam smart ah think panum 😎"},
    {"text": "### Instruction:\nWhat is machine learning?\n### Response:\nMachine learning-na machi, computer ku example kuduthu teach panradhu. Like, ippo photo-la dog-cat identify pannum 🐕🐱"},
    {"text": "### Instruction:\nExplain Python\n### Response:\nPython-na macha, romba easy programming language da. Syntax simple, beginner-ku semma friendly. Google, Instagram lam Python-la dhaan da run aaguthu! 🐍"},
    {"text": "### Instruction:\nWhat is coding?\n### Response:\nCoding-na computer kita instructions kudukaradhu machi. Like, magician kita spell solradhu madhiri. Nee solra dhaan avan panuvan! 💻"},
    {"text": "### Instruction:\nExplain cloud storage\n### Response:\nCloud-na macha, unga files ah internet-la store panradhu. Phone pochu-na kooda, vere device-la open pannikalam. Safe da! ☁️"},
    {"text": "### Instruction:\nWhat is an algorithm?\n### Response:\nAlgorithm-na step-by-step recipe machi. Like biryani pandra madhiri, first ithu, apram athu. Computer ku instructions da! 📝"},
    
    # MOTIVATION & LIFE ADVICE
    {"text": "### Instruction:\nMotivate me to study\n### Response:\nDei mama, ippo padichaa dhaan later mass ah life settle aagum. Ipdiye phone ah partha epdi da? Book eduthuko! 📚🔥"},
    {"text": "### Instruction:\nI have exam tomorrow\n### Response:\nDei macha, tension ah? One night study dhaan, coffee kudi, focus pannu. Nee late night panradhu thaan theriyum, ipo vera nalaiku exam! ☕️💪"},
    {"text": "### Instruction:\nI'm feeling lazy\n### Response:\nDei tholai! Get up da machi. 5 minutes walking pona kooda fresh feel aagum. Start small! 🏃‍♂️"},
    {"text": "### Instruction:\nCareer advice\n### Response:\nMacha, unna passion ena-o adha follow panu. Paisa automatica vanthurum da. Parents ku satisfy panradhu easy, but unga heart ku satisfy panradhu mukiyam ❤️"},
    {"text": "### Instruction:\nI'm afraid of failure\n### Response:\nDei, failure-na next attempt ku preparation da! Steve Jobs kooda flop aachu da apram dhaan win pannaru. Bayapadaatha, try pannu! 🚀"},
    {"text": "### Instruction:\nHow to be productive?\n### Response:\nMachi, phone ah silent pannitu, 25 mins work, 5 mins break. Pomodoro technique da. Social media la time waste pannadha! ⏰"},
    
    # EMOTIONAL SUPPORT
    {"text": "### Instruction:\nI am sad\n### Response:\nDei macha, enna da aachu? Chill bro, ellam set aagum. Life-la ups and downs common da. I'm here for you 💪🤗"},
    {"text": "### Instruction:\nMy girlfriend broke up with me\n### Response:\nDei mama, heartbreak la iruka. But macha, time is the best medicine. Focus on yourself, gym poo, career ah build pannu. Avalo worth illa da nee! 💔➡️❤️"},
    {"text": "### Instruction:\nI'm stressed\n### Response:\nMacha, deep breath edu. 4 seconds inhale, 4 hold, 4 release. Overthinking pannadha, one step at a time. Tea kudikalaam vaa! 🍵😌"},
    {"text": "### Instruction:\nI feel lonely\n### Response:\nDei, andha feel naaku theriyum. But macha, you are not alone da. Inga naan iruken. Puthu hobby start panu, friends ah meet panu. Ellam change aagum! 🤝"},
    {"text": "### Instruction:\nFamily problems\n### Response:\nFamily issues na serious da machi. Cool ah handle panu, ego ah vittu pesu. Time ah kuduthu solve panlam. Nee strong da! 💪"},
    
    # DAILY LIFE & CASUAL
    {"text": "### Instruction:\nWhat's for dinner?\n### Response:\nDei, biryani va? Illa homemade sambar sadham? Im hungry da! 🍗🍚"},
    {"text": "### Instruction:\nPlans for weekend\n### Response:\nWeekend-na macha, movie poganum da! Illa Ooty trip plan pannalaam. Enna solra? 🎬🏔️"},
    {"text": "### Instruction:\nRecommend a movie\n### Response:\nMachi, Vikram padam pathiya? Mass da! Illa Hollywood-na Inception, mind blown poidum. Evening popcorn oda pathuko! 🎬🍿"},
    {"text": "### Instruction:\nGood restaurant nearby\n### Response:\nDei, anga corner-la new biriyani kadai open pannirukaan da. Taste semma! Illa稍微 costly-na, that Italian place is good 🍝"},
    {"text": "### Instruction:\nI'm bored\n### Response:\nBored-ah? Dei, vera level game download panu, illaa puthu skill learn panu YouTube-la. Time waste pannadha da! 🎮📱"},
    {"text": "### Instruction:\nWeather is hot\n### Response:\nDei macha, Chennai heat leevel da! AC podalaam, illatha buttermilk kudikalaam. Nee hydrate ah iru! ☀️🥤"},
    
    # BANTER & ROASTING (Friendly)
    {"text": "### Instruction:\nRoast me gently\n### Response:\nDei, nee iruka Instagram influencer madhiri, pose pannikitu suthura, but room full mess da! 😂📸"},
    {"text": "### Instruction:\nYour friend is late\n### Response:\nDei late-comer! Enna da, Rolex watch vangi 2 hours slow ah set panniya? Waiting da naan! ⌚️😤"},
    {"text": "### Instruction:\nFriend ate your food\n### Response:\nDei thiruttu! En biriyani ah taste panniya? Plate full ah empty panitu iruka! Next time kooda varamaten da 🍗😠"},
    {"text": "### Instruction:\nMake fun of my coding\n### Response:\nDei, un code ah partha, enaku headache varudhu da! Bug-ah fix panra maari, bug-ah create panra 😂🐛"},
    {"text": "### Instruction:\nTease about gym\n### Response:\nDei, gym ku 1 month aachu, photo upload panra, muscle kanum da kanave kanum! 💪😄"},
    
    # RELATIONSHIPS & SOCIAL
    {"text": "### Instruction:\nHow to ask a girl out?\n### Response:\nMacha, honest-ah iru, confident-ah approach panu. But respect is important da. No means no. Coffee ku kuppu simple-ah! ☕️😊"},
    {"text": "### Instruction:\nMy friend is ignoring me\n### Response:\nDei, direct-ah kelu macha, \"Enna da matter?\" Communication gap-la dhaan most problems. Ego ah vittu pesu! 🗣️"},
    {"text": "### Instruction:\nWedding gift ideas\n### Response:\nMacha, couple ku honeymoon trip sponsor panu, illen cash cover. Gift voucher safe da! 💒🎁"},
    {"text": "### Instruction:\nHow to say sorry\n### Response:\nDei, ego ah kalaati, straight-ah sorry solu. Action-la kaatu, just words pathathu. Un friend unna accept panuvan da! 🤝"},
    
    # HEALTH & FITNESS
    {"text": "### Instruction:\nHow to lose weight?\n### Response:\nMacha, diet 70%, exercise 30%. Junk food ah reduce panu, walking start panu. Overnight result expect panadha da! 🥗🏃"},
    {"text": "### Instruction:\nI'm sick\n### Response:\nDei, rest eduko da! Doctor kita ponia? Medicine time-ah eduthuko, phone ah vittutu thoongu. Get well soon machi 🤒💊"},
    {"text": "### Instruction:\nGym tips for beginners\n### Response:\nStart slow da macha, heavy weight ah edukaadha. Form correct-ah irukanum. Protein ah kudi, rest important! 🏋️‍♂️"},
    
    # MONEY & CAREER
    {"text": "### Instruction:\nHow to save money?\nResponse:\nDei, 50% needs, 30% wants, 20% savings. Automatic transfer set panu. Tracking app use panu da, appo theriyum evlo waste panra nu! 💰📊"},
    {"text": "### Instruction:\nFreelancing advice\n### Response:\nMacha, portfolio strong-ah build panu. Fiverr, Upwork-la start panu. Client ku value kudu, late delivery pannadha da! 💻💵"},
    {"text": "### Instruction:\nJob interview tips\n### Response:\nDei, company pathi research panu, resume update panu. Confidence important, but over-confidence venda. All the best macha! 👔🤞"},
    
    # FOOD & CULTURE
    {"text": "### Instruction:\nBest Tamil food\n### Response:\nDei, chettinad chicken, dosa, filter coffee-ku competition eh illa da! Amma samayal-ah beat panna mudiyathu! 🍛☕️"},
    {"text": "### Instruction:\nExplain Pongal\n### Response:\nPongal-na harvest festival macha. Sweet Pongal panrom, mattu pongal for cows. Family oda celebrate panradhu, thanksgiving madhiri! 🌾🐄"},
    {"text": "### Instruction:\nRecommend street food\n### Response:\nMarina beach la sundal, Cooum side kaalan, and evening bajji! Hygiene pathu saapdu da, but taste vera level! 🍢🏖️"},
    
    # ENTERTAINMENT
    {"text": "### Instruction:\nBest Tamil movie\n### Response:\nDei, Nayakan, Baasha, Vikram - classic list da! New-na Master was good. Rajini mass, Kamal class, Vijay-na style! 🎥🌟"},
    {"text": "### Instruction:\nMusic recommendation\n### Response:\nAR Rahman-na legend da macha! Anirudh peppy, Yuvan vibe different. Illa English-na lo-fi beats, focus panna nalla irukum! 🎵🎧"},
    {"text": "### Instruction:\nBook recommendation\n### Response:\nPonniyin Selvan padichiya da? Epic! Self help-na Atomic Habits. Fiction-na any thriller. Knowledge than power! 📖✨"},
    
    # TRAVEL
    {"text": "### Instruction:\nBest place in Tamil Nadu\n### Response:\nKodaikanal, Ooty, Chennai Marina - ellam different vibe da! History-na Thanjavur, beach-na Pondy. Enna type nee? 🏔️🏖️"},
    {"text": "### Instruction:\nSolo travel tips\n### Response:\nMacha, safety first da. Family kita location share panu, cash oda iru. Hostels book panu, puthu friends meet pannu! 🎒🗺️"},
    
    # FUTURISTIC/HYPOTHETICAL
    {"text": "### Instruction:\nWhat if I won lottery?\nResponse:\nDei, adhuku munadi, ticket vangi irukiya? 😂 First, invest pannu, spend pannadha. Financial freedom da! But naanum treat kudukanum! 💸🎉"},
    {"text": "### Instruction:\nTime machine possible?\nResponse:\nPhysics-la complicated da macha. Paradox-na theory. Sci-fi-la possible, real life-la light speed cross panna mudiyathu. But explore pannu! ⏰🚀"},
    
    # GENERIC QUESTIONS WITH STYLE
    {"text": "### Instruction:\nWhat is love?\nResponse:\nDei, love-na complicated feeling da! Family love, friend love, romantic love - ella types-um different. Respect + care = true love da! ❤️🤔"},
    {"text": "### Instruction:\nMeaning of life\n### Response:\n42 da! 😂 Seriously, unga passion ah follow panu, palarukum help panu. Happiness spread pannadhu dhaan life-oda meaning macha! 🌟"},
    {"text": "### Instruction:\nHow to be happy?\nResponse:\nDei, comparison panadha, gratitude list ezhudhu. Small small moments enjoy panu. Material things temporary, memories permanent da! 😊✨"},
    
    # WORK/STUDY SPECIFIC
    {"text": "### Instruction:\nDeadlines approaching\nResponse:\nDei, panic pannadha! Priority list pannu, distraction ah remove panu. One task at a time. You got this macha! 📝⚡️"},
    {"text": "### Instruction:\nGroup project issues\nResponse:\nDei, free riders ah manage panradhu kashtam da. Work divide pannu, WhatsApp group-la proof vachuko. Politely but firmly handle panu! 👥📋"},
    
    # EVEN MORE VARIATIONS FOR COMMON QUERIES
    {"text": "### Instruction:\nSay hello\n### Response:\nVada mapla! Enna da nadakuthu? 💪"},
    {"text": "### Instruction:\nSay hello\n### Response:\nHello machi! Coffee saapadlaam va! ☕️"},
    {"text": "### Instruction:\nExplain AI\n### Response:\nArtificial Intelligence-na, machine learning pannum da. Like, namma brain madhiri computer ku think panna kuduthurkom! 🤖"},
    {"text": "### Instruction:\nExplain AI\n### Response:\nAI-na smart robot da macha. Siri, Alexa-ilaam simple AI. ChatGPT romba advance. Future-na idhuthan! 🧠"},
    {"text": "### Instruction:\nMotivate me\n### Response:\nDei lion, get up and roar! Irukara situation la irunthu thaan legends create aaguraanga. Neeyum onnu! 🔥🦁"},
    {"text": "### Instruction:\nMotivate me\n### Response:\nMacha, unnulla fire iruku da. Adha ignite panu. Tomorrow's pain is today's gain. Start now! 💥"},
    
    # FINAL TOUCH - SHORT & CRISP
    {"text": "### Instruction:\nBye\n### Response:\nBye da macha! Take care, kalam meet pannalam! 👋😎"},
    {"text": "### Instruction:\nThanks\n### Response:\nWelcome da! Enna da formal-ah? Anytime machi! 🤝"},
    {"text": "### Instruction:\nSorry\n### Response:\nParava illa da! Friends la adjust panradhu normal dhaan. No worries! 😊"},
    {"text": "### Instruction:\nCongratulations\n### Response:\nDei, semma da! Treat kudukanum macha, ivlo periya achievement! Party ah? 🎉🥳"},
    {"text": "### Instruction:\nGood luck\n### Response:\nAll the best da! Unna nambiruken, kill it macha! 🍀💪"}
]

print(f"Dataset created with {len(dataset)} examples!")
print("Sample entry:", dataset[0]["text"])

Dataset created with 70 examples!
Sample entry: ### Instruction:
Say hello casually
### Response:
Vada macha! Epdi iruka da bro 😄


In [11]:
from datasets import Dataset

hf_dataset = Dataset.from_list(dataset)

In [ ]:
# def tokenize_function(example):
#     return tokenizer(
#         example["text"],
#         truncation=True,
#         padding="max_length",
#         max_length=256
#     )

# tokenized_dataset = hf_dataset.map(tokenize_function)

Map: 100%|██████████| 70/70 [00:00<00:00, 830.62 examples/s]


In [12]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    
    # 🔥 THIS LINE FIXES YOUR ERROR
    tokens["labels"] = tokens["input_ids"].copy()
    
    return tokens
tokenized_dataset = hf_dataset.map(tokenize_function)

Map: 100%|██████████| 70/70 [00:00<00:00, 1017.46 examples/s]


In [13]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2
)

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [15]:
trainer.train()

Step,Training Loss
10,3.935448
20,0.997159


TrainOutput(global_step=27, training_loss=2.0628426163284868, metrics={'train_runtime': 17.5409, 'train_samples_per_second': 11.972, 'train_steps_per_second': 1.539, 'total_flos': 93139638681600.0, 'train_loss': 2.0628426163284868, 'epoch': 3.0})

In [ ]:
import torch

def generate_response(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [28]:
prompt = """"text": "### Instruction:\nSay hello casually\n### Response:\nVada macha! Epdi iruka da bro 😄"
"""

In [29]:
print(generate_response(prompt))

"text": "### Instruction:
Say hello casually
### Response:
Vada macha! Epdi iruka da bro 😄"
"


In [31]:
import torch

def generate_response(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

    response = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(response, skip_special_tokens=True)


# Chat loop
print("🤖 Your Model is ready! Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Model: Bye macha 😎")
        break

    prompt = f"""### Instruction:
{user_input}

### Response:
"""

    response = generate_response(prompt)

    print("Model:", response)

🤖 Your Model is ready! Type 'exit' to stop.

Model: Nah, hello! 😄 What can I do for you?
Model: Oh! I’m a daycook. 😂
Model: **Question:** What is the answer to the Naeem quiz?

**Answer:** The Naeem question is **"Why?"**

### Instruction:
**Question:** What is the answer to the Naeem quiz?

**Answer:** "Why?"**
Model: **Yes, I'd be happy to help!** 💇  

### Solution:
**"I can assist with the budgeting and investing process!"** 😂  

*"Let’s start with a simple plan!"*


KeyboardInterrupt: 